# Nube de Palabras — Carpeta de documentos

**Extensión del proyecto `4/main.ipynb`**: en lugar de abrir un archivo individual, el usuario selecciona una **carpeta completa**. La aplicación encuentra todos los documentos compatibles, los lee en paralelo y genera una nube de palabras unificada con el texto combinado.

---

## Diferencias respecto a `4/main.ipynb`

| Característica | `4/` (archivo único) | `5/` (carpeta) |
|---------------|----------------------|----------------|
| Selección | Un archivo | Una carpeta entera |
| Búsqueda | — | Recursiva (subcarpetas opcionales) |
| Procesamiento | Secuencial | Paralelo con `ThreadPoolExecutor` |
| Progreso | Barra indeterminada | Barra determinada + `N/Total archivos` |
| Panel lateral | Solo opciones | Opciones + **lista de archivos** con estado |
| Texto base | Un documento | Concatenación de todos los documentos |

## Formatos soportados

`.txt` · `.md` · `.py` · `.xml` · `.log` · `.pdf` · `.docx` · `.json` · `.csv` · `.html` · `.htm`

---
##  Heurísticas de Nielsen — Cambios específicos para carpetas

Las 10 heurísticas se mantienen igual que en `4/`. Los cambios específicos al modo carpeta son:

**H1 · Visibilidad del estado**  
La barra de progreso ahora es **determinada** (`value=N/total`) y muestra el nombre del archivo que se está leyendo en tiempo real. Un panel colapsable lista todos los archivos con su estado: ` pendiente`, ` leído`, ` error`.

**H3 · Control y libertad**  
El usuario puede **deseleccionar archivos individuales** de la lista antes de generar la nube. Si un archivo falla, el resto continúa procesándose.

**H5 · Prevención de errores**  
- Solo se muestran archivos con extensión soportada (los demás se ignoran silenciosamente).  
- Se muestra un aviso si la carpeta está vacía o no contiene documentos compatibles.  
- Opción de búsqueda recursiva activable para evitar procesar carpetas no deseadas.

**H1 + H9 · Estado + Recuperación**  
Al terminar, se muestra un resumen: `X archivos leídos, Y errores`. Los errores se listan con su causa específica.

---
##  1. Instalación de dependencias

In [1]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'wordcloud', 'pillow', 'PyPDF2', 'python-docx',
    'matplotlib', 'nltk', 'pdfplumber', 'beautifulsoup4',
    '-q'
], check=True)
print(' Dependencias instaladas')

 Dependencias instaladas



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


---
##  2. Imports

In [2]:
#  Estándar 
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, colorchooser
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
import csv
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

#  Visualización 
from wordcloud import WordCloud, STOPWORDS
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

#  NLP 
import nltk
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

#  Lectura de formatos (detección de disponibilidad) 
try:
    import pdfplumber;    HAS_PDFPLUMBER = True
except ImportError:       HAS_PDFPLUMBER = False
try:
    import PyPDF2;        HAS_PYPDF2 = True
except ImportError:       HAS_PYPDF2 = False
try:
    from docx import Document as DocxDocument; HAS_DOCX = True
except ImportError:       HAS_DOCX = False
try:
    from bs4 import BeautifulSoup; HAS_BS4 = True
except ImportError:       HAS_BS4 = False

print('pdfplumber:', HAS_PDFPLUMBER, '| PyPDF2:', HAS_PYPDF2,
      '| python-docx:', HAS_DOCX, '| bs4:', HAS_BS4)
print(' Imports completados')

pdfplumber: True | PyPDF2: True | python-docx: True | bs4: True
 Imports completados


---
##  3. Módulo de lectura de archivos

Mismo módulo que `4/main.ipynb`, sin cambios.

In [3]:
def read_txt(path):
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            with open(path, 'r', encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
    raise ValueError(f'No se pudo decodificar: {path}')

def read_pdf(path):
    if HAS_PDFPLUMBER:
        with pdfplumber.open(path) as pdf:
            return '\n'.join(p.extract_text() or '' for p in pdf.pages)
    if HAS_PYPDF2:
        with open(path, 'rb') as f:
            r = PyPDF2.PdfReader(f)
            return '\n'.join(p.extract_text() or '' for p in r.pages)
    raise ImportError('Instala pdfplumber o PyPDF2.')

def read_docx(path):
    if not HAS_DOCX:
        raise ImportError('Instala python-docx.')
    return '\n'.join(p.text for p in DocxDocument(path).paragraphs)

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    texts = []
    def _ex(obj):
        if isinstance(obj, str):   texts.append(obj)
        elif isinstance(obj, dict): [_ex(v) for v in obj.values()]
        elif isinstance(obj, list): [_ex(i) for i in obj]
    _ex(data)
    return ' '.join(texts)

def read_csv(path):
    texts = []
    for enc in ('utf-8', 'latin-1'):
        try:
            with open(path, 'r', encoding=enc, newline='') as f:
                [texts.extend(r) for r in csv.reader(f)]
            break
        except UnicodeDecodeError:
            continue
    return ' '.join(texts)

def read_html(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
    if HAS_BS4:
        return BeautifulSoup(content, 'html.parser').get_text(' ')
    return re.sub(r'<[^>]+>', ' ', content)

FILE_READERS = {
    '.txt': read_txt, '.md': read_txt,  '.py':  read_txt,
    '.xml': read_txt, '.log': read_txt,
    '.pdf': read_pdf,
    '.docx': read_docx,
    '.json': read_json,
    '.csv':  read_csv,
    '.html': read_html, '.htm': read_html,
}
SUPPORTED_EXTENSIONS = set(FILE_READERS.keys())

def read_file(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext not in FILE_READERS:
        raise ValueError(f'Formato no soportado: {ext}')
    return FILE_READERS[ext](path)

print(' Lectores de archivo cargados —', ', '.join(sorted(SUPPORTED_EXTENSIONS)))

 Lectores de archivo cargados — .csv, .docx, .htm, .html, .json, .log, .md, .pdf, .py, .txt, .xml


---
##  4. Módulo de exploración de carpetas

Nuevo módulo específico de esta versión. `scan_folder` recorre una carpeta (opcionalmente en profundidad) y devuelve la lista de rutas con extensión soportada.

El modelo de datos `FileEntry` almacena el estado de procesamiento de cada archivo para mostrarlo en la lista de la UI.

In [4]:
@dataclass
class FileEntry:
    """Estado de procesamiento de un archivo individual."""
    path:    Path
    enabled: bool  = True   # el usuario puede deseleccionarlo (H3)
    status:  str   = 'pending'   # pending | ok | error
    error:   str   = ''
    words:   int   = 0

    @property
    def name(self) -> str:
        return self.path.name

    @property
    def icon(self) -> str:
        return {'pending': '', 'ok': '', 'error': ''}.get(self.status, '')


def scan_folder(folder: str, recursive: bool = False) -> list[FileEntry]:
    """
    Recorre la carpeta y devuelve FileEntry por cada archivo soportado.
    recursive=True incluye subcarpetas.
    """
    root = Path(folder)
    glob = root.rglob('*') if recursive else root.glob('*')
    entries = [
        FileEntry(path=p)
        for p in sorted(glob)
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    return entries


def read_entries_parallel(
    entries: list[FileEntry],
    on_progress,          # callback(entry, idx, total) llamado desde el hilo
    max_workers: int = 4,
) -> tuple[str, list[FileEntry]]:
    """
    Lee en paralelo todos los FileEntry habilitados.
    Devuelve (texto_combinado, lista_actualizada).
    """
    enabled = [e for e in entries if e.enabled]
    total   = len(enabled)
    texts   = [''] * total

    def _read(idx_entry):
        idx, entry = idx_entry
        try:
            text = read_file(str(entry.path))
            entry.words  = len(text.split())
            entry.status = 'ok'
            texts[idx]   = text
        except Exception as exc:
            entry.status = 'error'
            entry.error  = str(exc)
        on_progress(entry, idx + 1, total)
        return idx

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(_read, (i, e)) for i, e in enumerate(enabled)]
        for f in as_completed(futures):
            f.result()  # propaga excepciones inesperadas

    combined = ' '.join(t for t in texts if t)
    return combined, entries


print(' Módulo de exploración de carpetas cargado')

 Módulo de exploración de carpetas cargado


---
##  5. Módulo generador de nube

Idéntico a `4/main.ipynb`.

In [5]:
STOPWORDS_LANGS = {
    'Español': 'spanish', 'Inglés': 'english', 'Francés': 'french',
    'Alemán':  'german',  'Italiano': 'italian', 'Portugués': 'portuguese',
    'Ninguno': None,
}
COLOR_SCHEMES = {
    'Viridis': 'viridis', 'Plasma':   'plasma',  'Inferno':  'inferno',
    'Magma':   'magma',   'Coolwarm': 'coolwarm', 'Blues':    'Blues',
    'Reds':    'Reds',    'Greens':   'Greens',
}

def _build_stopwords(lang_key, custom):
    sw = set(STOPWORDS)
    lang = STOPWORDS_LANGS.get(lang_key)
    if lang:
        try: sw |= set(stopwords.words(lang))
        except OSError: pass
    return sw | custom

def generate_wordcloud(
    text, max_words=100, bg_color='white', colormap='viridis',
    lang='Español', custom_stopwords=None, width=800, height=500,
):
    if not text or not text.strip():
        raise ValueError('El texto combinado está vacío.')
    sw = _build_stopwords(lang, custom_stopwords or set())
    clean = re.sub(r'[^a-záéíóúüñA-ZÁÉÍÓÚÜÑ\s]', ' ', text)
    return WordCloud(
        width=width, height=height, background_color=bg_color,
        max_words=max_words, stopwords=sw, colormap=colormap,
        collocations=True, min_font_size=10, max_font_size=100,
        prefer_horizontal=0.9, regexp=r'\b[^\d\W]{3,}\b',
    ).generate(clean)

print(' Generador de nube cargado')

 Generador de nube cargado


---
##  6. Widget Tooltip

In [6]:
class Tooltip:
    def __init__(self, widget, text):
        self._w, self._text, self._win = widget, text, None
        widget.bind('<Enter>', self._show)
        widget.bind('<Leave>', self._hide)

    def _show(self, _=None):
        if self._win: return
        x = self._w.winfo_rootx() + 20
        y = self._w.winfo_rooty() + self._w.winfo_height() + 4
        self._win = tw = tk.Toplevel(self._w)
        tw.wm_overrideredirect(True)
        tw.wm_geometry(f'+{x}+{y}')
        tk.Label(tw, text=self._text, justify='left', background='#FFFFE0',
                 relief='solid', borderwidth=1, font=('Segoe UI', 9),
                 wraplength=260, padx=5, pady=3).pack()

    def _hide(self, _=None):
        if self._win: self._win.destroy(); self._win = None

print(' Tooltip cargado')

 Tooltip cargado


---
##  7. Aplicación principal — `FolderWordCloudApp`

### Layout

```

  BARRA SUPERIOR    Nube de Palabras · Carpeta              

  PANEL IZQ. 300px                                           
                            ÁREA DE VISUALIZACIÓN            
  CARPETA                   (matplotlib canvas)              
  [Seleccionar…]                                             
   Subcarpetas                                              
                                                             
  ARCHIVOS (lista)                                           
   doc1.pdf   120w                                          
   notas.txt   88w                                          
   roto.pdf  error                                          
                                                             
  OPCIONES                                                   
  EXCLUIR PALABRAS                                           
  [Generar] [Guardar                                         
  [Deshacer]                                                 

  BARRA DE ESTADO    //   mensaje          N palabras     

```

### Flujo de carpeta
```
open_folder()
   scan_folder()              lista FileEntry
   _populate_file_list()      muestra archivos con checkbox

generate()
   hilo: read_entries_parallel()
       on_progress callback   actualiza barra determinada + lista
   texto combinado  generate_wordcloud()
   render matplotlib
   resumen: X ok, Y errores
```

In [7]:
class FolderWordCloudApp(tk.Tk):
    """Nube de palabras a partir de una carpeta de documentos."""

    THEME = {
        'bg':          '#F5F7FA',
        'panel':       '#FFFFFF',
        'accent':      '#4A90D9',
        'accent_dark': '#2D6EAA',
        'text':        '#2C3E50',
        'subtext':     '#7F8C8D',
        'border':      '#E0E4E8',
        'ok':          '#27AE60',
        'error':       '#E74C3C',
    }

    def __init__(self):
        super().__init__()
        self.title('Nube de Palabras · Carpeta')
        self.geometry('1280x780')
        self.minsize(960, 620)
        self.configure(bg=self.THEME['bg'])

        self._entries:      list[FileEntry]     = []
        self._check_vars:   list[tk.BooleanVar] = []
        self._wordcloud:    Optional[WordCloud] = None
        self._history:      list[WordCloud]     = []
        self._processing:   bool                = False
        self._combined_text: str                = ''

        self._setup_style()
        self._build_ui()

        # Atajos (H7)
        self.bind_all('<Control-o>', lambda e: self.open_folder())
        self.bind_all('<Control-g>', lambda e: self.generate())
        self.bind_all('<Control-s>', lambda e: self.save_image())
        self.bind_all('<Control-z>', lambda e: self.undo())
        self.bind_all('<F5>',        lambda e: self.generate())

        self._set_status('Bienvenido — Selecciona una carpeta para comenzar  (Ctrl+O)', 'info')
        self.protocol('WM_DELETE_WINDOW', lambda: (plt.close('all'), self.destroy()))

    #  Estilos 

    def _setup_style(self):
        s = ttk.Style(self)
        s.theme_use('clam')
        T = self.THEME
        s.configure('TFrame',        background=T['bg'])
        s.configure('TLabel',        background=T['panel'], foreground=T['text'],
                    font=('Segoe UI', 10))
        s.configure('Sub.TLabel',    background=T['panel'], foreground=T['subtext'],
                    font=('Segoe UI', 8))
        s.configure('BG.TLabel',     background=T['bg'],    foreground=T['subtext'],
                    font=('Segoe UI', 8))
        s.configure('Accent.TButton', background=T['accent'], foreground='white',
                    font=('Segoe UI', 10, 'bold'), padding=(12, 6))
        s.map('Accent.TButton',
              background=[('active', T['accent_dark']), ('disabled', '#B0BEC5')])
        s.configure('TButton',       font=('Segoe UI', 9), padding=(8, 4))
        s.configure('TCombobox',     font=('Segoe UI', 9))
        s.configure('TSpinbox',      font=('Segoe UI', 9))
        s.configure('TCheckbutton',  background=T['panel'], foreground=T['text'],
                    font=('Segoe UI', 9))
        s.configure('TProgressbar',  troughcolor=T['border'],
                    background=T['accent'], thickness=7)

    #  UI 

    def _build_ui(self):
        self._build_top_bar()
        content = tk.Frame(self, bg=self.THEME['bg'])
        content.pack(fill='both', expand=True)
        self._build_left_panel(content)
        self._build_right_panel(content)
        self._build_status_bar()

    def _build_top_bar(self):
        T = self.THEME
        bar = tk.Frame(self, bg=T['accent'], height=52)
        bar.pack(fill='x')
        bar.pack_propagate(False)
        tk.Label(bar, text='  Nube de Palabras · Carpeta',
                 bg=T['accent'], fg='white',
                 font=('Segoe UI', 15, 'bold')).pack(side='left', padx=16)
        tk.Label(bar,
                 text='Ctrl+O Abrir carpeta · Ctrl+G Generar · Ctrl+S Guardar · Ctrl+Z Deshacer',
                 bg=T['accent'], fg='#D6EAF8',
                 font=('Segoe UI', 8)).pack(side='right', padx=16)

    def _build_left_panel(self, parent):
        T = self.THEME
        left = tk.Frame(parent, bg=T['panel'], width=300,
                        highlightbackground=T['border'], highlightthickness=1)
        left.pack(side='left', fill='y')
        left.pack_propagate(False)

        # Scroll interior
        canvas_scroll = tk.Canvas(left, bg=T['panel'], highlightthickness=0)
        scrollbar = ttk.Scrollbar(left, orient='vertical', command=canvas_scroll.yview)
        canvas_scroll.configure(yscrollcommand=scrollbar.set)
        scrollbar.pack(side='right', fill='y')
        canvas_scroll.pack(side='left', fill='both', expand=True)

        self._inner = tk.Frame(canvas_scroll, bg=T['panel'])
        win_id = canvas_scroll.create_window((0, 0), window=self._inner, anchor='nw')

        def _on_configure(e):
            canvas_scroll.configure(scrollregion=canvas_scroll.bbox('all'))
            canvas_scroll.itemconfig(win_id, width=canvas_scroll.winfo_width())
        self._inner.bind('<Configure>', _on_configure)
        canvas_scroll.bind('<Configure>', _on_configure)
        canvas_scroll.bind_all('<MouseWheel>',
            lambda e: canvas_scroll.yview_scroll(-1*(e.delta//120), 'units'))

        self._build_panel_content(self._inner)

    def _build_panel_content(self, parent):
        T = self.THEME
        pad = {'padx': 14}

        #  Carpeta 
        ttk.Label(parent, text='CARPETA', style='Sub.TLabel').pack(anchor='w', pady=(14,0), **pad)

        self._folder_var = tk.StringVar(value='Ninguna carpeta seleccionada')
        folder_box = tk.Frame(parent, bg='#F8F9FA',
                              highlightbackground=T['border'], highlightthickness=1)
        folder_box.pack(fill='x', pady=(4, 6), **pad)
        tk.Label(folder_box, textvariable=self._folder_var, bg='#F8F9FA',
                 fg=T['subtext'], font=('Segoe UI', 8), wraplength=240,
                 justify='left', pady=5, padx=6).pack(fill='x')

        self._recursive_var = tk.BooleanVar(value=False)
        rec_cb = ttk.Checkbutton(parent, text='Incluir subcarpetas',
                                 variable=self._recursive_var)
        rec_cb.pack(anchor='w', pady=(0, 4), **pad)
        Tooltip(rec_cb, 'Busca documentos en todas las subcarpetas de forma recursiva')

        open_btn = ttk.Button(parent, text='  Seleccionar carpeta…',
                              command=self.open_folder)
        open_btn.pack(fill='x', pady=(0, 4), **pad)
        Tooltip(open_btn, 'Abrir carpeta con documentos (Ctrl+O)')

        ttk.Separator(parent, orient='horizontal').pack(fill='x', pady=8, **pad)

        #  Lista de archivos 
        hdr = tk.Frame(parent, bg=T['panel'])
        hdr.pack(fill='x', **pad)
        ttk.Label(hdr, text='ARCHIVOS', style='Sub.TLabel').pack(side='left')
        self._file_count_lbl = ttk.Label(hdr, text='', style='Sub.TLabel')
        self._file_count_lbl.pack(side='right')

        # Botones seleccionar/deseleccionar todos (H3)
        sel_row = tk.Frame(parent, bg=T['panel'])
        sel_row.pack(fill='x', pady=(2,4), **pad)
        ttk.Button(sel_row, text='Todos',
                   command=lambda: self._set_all_checks(True)).pack(side='left')
        ttk.Button(sel_row, text='Ninguno',
                   command=lambda: self._set_all_checks(False)).pack(side='left', padx=4)
        Tooltip(sel_row, 'Seleccionar o deseleccionar todos los archivos')

        # Contenedor de la lista (se rellena en _populate_file_list)
        self._file_list_frame = tk.Frame(parent, bg=T['panel'])
        self._file_list_frame.pack(fill='x', **pad)

        ttk.Separator(parent, orient='horizontal').pack(fill='x', pady=8, **pad)

        #  Opciones 
        ttk.Label(parent, text='OPCIONES', style='Sub.TLabel').pack(anchor='w', **pad)

        r1 = tk.Frame(parent, bg=T['panel'])
        r1.pack(fill='x', pady=3, **pad)
        ttk.Label(r1, text='Máx. palabras').pack(side='left')
        self._max_words = tk.IntVar(value=150)
        sp = ttk.Spinbox(r1, from_=10, to=500, increment=10,
                         textvariable=self._max_words, width=6)
        sp.pack(side='right')
        Tooltip(sp, 'Número máximo de palabras en la nube (10–500)')

        r2 = tk.Frame(parent, bg=T['panel'])
        r2.pack(fill='x', pady=3, **pad)
        ttk.Label(r2, text='Idioma').pack(side='left')
        self._lang = tk.StringVar(value='Español')
        cb = ttk.Combobox(r2, textvariable=self._lang, width=10,
                          values=list(STOPWORDS_LANGS.keys()), state='readonly')
        cb.pack(side='right')
        Tooltip(cb, 'Idioma para filtrar palabras vacías')

        r3 = tk.Frame(parent, bg=T['panel'])
        r3.pack(fill='x', pady=3, **pad)
        ttk.Label(r3, text='Esquema de color').pack(side='left')
        self._colormap = tk.StringVar(value='Plasma')
        cb2 = ttk.Combobox(r3, textvariable=self._colormap, width=10,
                           values=list(COLOR_SCHEMES.keys()), state='readonly')
        cb2.pack(side='right')
        Tooltip(cb2, 'Paleta de colores para la nube')

        r4 = tk.Frame(parent, bg=T['panel'])
        r4.pack(fill='x', pady=3, **pad)
        ttk.Label(r4, text='Fondo').pack(side='left')
        self._bg_color = tk.StringVar(value='white')
        self._bg_swatch = tk.Label(r4, bg='white', width=4, height=1,
                                   relief='solid', bd=1, cursor='hand2')
        self._bg_swatch.pack(side='right')
        self._bg_swatch.bind('<Button-1>', self._pick_bg)
        Tooltip(self._bg_swatch, 'Clic para elegir color de fondo')

        ttk.Separator(parent, orient='horizontal').pack(fill='x', pady=8, **pad)

        ttk.Label(parent, text='EXCLUIR PALABRAS', style='Sub.TLabel').pack(anchor='w', **pad)
        ttk.Label(parent, text='(separadas por coma)', style='Sub.TLabel').pack(anchor='w', **pad)
        self._custom_sw = tk.Text(parent, height=3, font=('Segoe UI', 9),
                                  relief='solid', bd=1, wrap='word', bg='#FAFAFA')
        self._custom_sw.pack(fill='x', pady=(4, 8), **pad)
        Tooltip(self._custom_sw, 'Ej: empresa, año, dato, valor')

        ttk.Separator(parent, orient='horizontal').pack(fill='x', pady=4, **pad)

        #  Botones de acción 
        gen_btn = ttk.Button(parent, text='  Generar nube  (F5)',
                             command=self.generate, style='Accent.TButton')
        gen_btn.pack(fill='x', pady=(8, 4), **pad)
        Tooltip(gen_btn, 'Leer archivos seleccionados y generar nube combinada (Ctrl+G)')

        save_btn = ttk.Button(parent, text='  Guardar imagen…', command=self.save_image)
        save_btn.pack(fill='x', pady=2, **pad)
        Tooltip(save_btn, 'Exportar la nube como PNG, JPG, SVG o PDF (Ctrl+S)')

        undo_btn = ttk.Button(parent, text='  Deshacer  (Ctrl+Z)', command=self.undo)
        undo_btn.pack(fill='x', pady=2, **pad)
        Tooltip(undo_btn, 'Restaurar la nube anterior (Ctrl+Z)')

        # Progreso determinado (H1 — novedad vs. versión 4)
        self._progress_var = tk.DoubleVar(value=0)
        self._progress = ttk.Progressbar(parent, variable=self._progress_var,
                                         maximum=100, mode='determinate')
        self._progress.pack(fill='x', pady=(10, 0), **pad)
        self._progress_lbl = ttk.Label(parent, text='', style='Sub.TLabel')
        self._progress_lbl.pack(anchor='w', **pad)

        tk.Frame(parent, bg=T['panel'], height=14).pack()

    def _build_right_panel(self, parent):
        T = self.THEME
        right = tk.Frame(parent, bg=T['bg'])
        right.pack(side='left', fill='both', expand=True)

        self._canvas_frame = tk.Frame(right, bg=T['bg'])
        self._canvas_frame.pack(fill='both', expand=True, padx=16, pady=16)

        # Placeholder
        self._placeholder = tk.Frame(self._canvas_frame, bg='white',
                                     highlightbackground=T['border'], highlightthickness=1)
        self._placeholder.pack(fill='both', expand=True)
        ph = tk.Frame(self._placeholder, bg='white')
        ph.place(relx=0.5, rely=0.5, anchor='center')
        tk.Label(ph, text='', bg='white', fg=T['border'],
                 font=('Segoe UI', 64)).pack()
        tk.Label(ph, text='Selecciona una carpeta y haz clic en Generar',
                 bg='white', fg=T['subtext'], font=('Segoe UI', 13)).pack()
        tk.Label(ph, text='Formatos: TXT · PDF · DOCX · JSON · CSV · HTML · MD · XML · LOG · PY',
                 bg='white', fg='#B0BEC5', font=('Segoe UI', 9)).pack(pady=4)

        # Canvas matplotlib
        self._fig, self._ax = plt.subplots(figsize=(9, 5.5))
        self._fig.patch.set_facecolor(T['bg'])
        self._ax.axis('off')
        self._mpl_canvas = FigureCanvasTkAgg(self._fig, master=self._canvas_frame)
        self._mpl_widget = self._mpl_canvas.get_tk_widget()

    def _build_status_bar(self):
        T = self.THEME
        bar = tk.Frame(self, bg='#ECF0F1',
                       highlightbackground=T['border'], highlightthickness=1)
        bar.pack(fill='x', side='bottom')
        self._status_icon = tk.StringVar(value='')
        self._status_var  = tk.StringVar(value='')
        tk.Label(bar, textvariable=self._status_icon, bg='#ECF0F1',
                 font=('Segoe UI', 10), width=2).pack(side='left', padx=(6,0), pady=3)
        tk.Label(bar, textvariable=self._status_var, bg='#ECF0F1',
                 fg=T['text'], font=('Segoe UI', 9), anchor='w').pack(side='left')
        self._word_count_var = tk.StringVar(value='')
        tk.Label(bar, textvariable=self._word_count_var, bg='#ECF0F1',
                 fg=T['subtext'], font=('Segoe UI', 8)).pack(side='right', padx=8)

    #  Lista de archivos 

    def _populate_file_list(self):
        """Dibuja la lista de archivos con checkboxes. (H3, H6)"""
        T = self.THEME
        for w in self._file_list_frame.winfo_children():
            w.destroy()
        self._check_vars.clear()

        if not self._entries:
            tk.Label(self._file_list_frame, text='No se encontraron archivos compatibles.',
                     bg=T['panel'], fg=T['subtext'], font=('Segoe UI', 8),
                     wraplength=240).pack(anchor='w', pady=4)
            return

        for entry in self._entries:
            var = tk.BooleanVar(value=entry.enabled)
            self._check_vars.append(var)

            row = tk.Frame(self._file_list_frame, bg=T['panel'])
            row.pack(fill='x', pady=1)

            cb = tk.Checkbutton(row, variable=var, bg=T['panel'],
                                activebackground=T['panel'],
                                command=lambda e=entry, v=var: setattr(e, 'enabled', v.get()))
            cb.pack(side='left')

            # Nombre del archivo (truncado)
            name = entry.name if len(entry.name) <= 26 else entry.name[:23] + '…'
            lbl = tk.Label(row, text=name, bg=T['panel'], fg=T['text'],
                           font=('Segoe UI', 8), anchor='w', width=22)
            lbl.pack(side='left')
            Tooltip(lbl, str(entry.path))

            # Etiqueta de estado (se actualiza durante el proceso)
            entry._status_lbl = tk.Label(row, text='', bg=T['panel'],
                                         font=('Segoe UI', 8), fg=T['subtext'])
            entry._status_lbl.pack(side='right')

        n = len(self._entries)
        self._file_count_lbl.configure(text=f'{n} archivo{'s' if n != 1 else ''}')

    def _update_entry_label(self, entry: FileEntry):
        """Actualiza el icono de estado de un archivo en la lista. (H1)"""
        if not hasattr(entry, '_status_lbl'):
            return
        T = self.THEME
        color = {'ok': T['ok'], 'error': T['error'], 'pending': T['subtext']}
        label = entry.icon
        if entry.status == 'ok':
            label += f' {entry.words:,}w'
        entry._status_lbl.configure(text=label, fg=color.get(entry.status, T['subtext']))

    def _set_all_checks(self, value: bool):
        for var, entry in zip(self._check_vars, self._entries):
            var.set(value)
            entry.enabled = value

    #  Manejadores principales 

    def open_folder(self):
        """Abre selector de carpeta, escanea archivos. (H3, H5)"""
        folder = filedialog.askdirectory(title='Seleccionar carpeta de documentos')
        if not folder:
            return

        recursive = self._recursive_var.get()
        entries = scan_folder(folder, recursive=recursive)

        if not entries:
            # H9: error descriptivo con solución sugerida
            messagebox.showwarning(
                'Carpeta vacía',
                'No se encontraron archivos compatibles en la carpeta seleccionada.\n\n'
                'Formatos aceptados: ' + ', '.join(sorted(SUPPORTED_EXTENSIONS)) +
                ('\n\nActiva «Incluir subcarpetas» para buscar en profundidad.' if not recursive else ''),
                parent=self
            )
            return

        self._entries = entries
        self._combined_text = ''

        # Nombre corto de la carpeta
        p = Path(folder)
        display = str(p) if len(str(p)) <= 38 else '…/' + p.name
        self._folder_var.set(display)

        self._populate_file_list()
        self._progress_var.set(0)
        self._progress_lbl.configure(text='')

        n = len(entries)
        self._set_status(
            f'{n} archivo{"s" if n>1 else ""} encontrado{"s" if n>1 else ""} '
            f'en «{p.name}». Haz clic en Generar.', 'success'
        )
        self._word_count_var.set(f'{n} archivos')

    def generate(self):
        """Lee todos los archivos habilitados y genera la nube. (H1, H3, H5)"""
        if self._processing:
            return
        if not self._entries:
            messagebox.showwarning('Sin carpeta',
                'Primero selecciona una carpeta con Ctrl+O.', parent=self)
            return

        enabled = [e for e in self._entries if e.enabled]
        if not enabled:
            messagebox.showwarning('Sin archivos',
                'No hay archivos seleccionados. Marca al menos uno.', parent=self)
            return

        # Guardar estado anterior para undo (H3)
        if self._wordcloud is not None:
            self._history.append(self._wordcloud)
            if len(self._history) > 10:
                self._history.pop(0)

        # Resetear estados visuales
        for e in self._entries:
            e.status = 'pending'
            if hasattr(e, '_status_lbl'):
                e._status_lbl.configure(text='', fg=self.THEME['subtext'])

        self._processing = True
        self._progress_var.set(0)
        self._set_status(f'Leyendo {len(enabled)} archivos…', 'info')

        # Opciones
        max_words = self._max_words.get()
        lang      = self._lang.get()
        colormap  = COLOR_SCHEMES.get(self._colormap.get(), 'plasma')
        bg_color  = self._bg_color.get()
        raw_sw    = self._custom_sw.get('1.0', 'end').strip()
        custom_sw = {w.strip().lower() for w in raw_sw.split(',') if w.strip()}
        entries   = list(self._entries)
        total_en  = len(enabled)
        ww = max(self._mpl_widget.winfo_width(),  600)
        wh = max(self._mpl_widget.winfo_height(), 400)

        def _on_progress(entry: FileEntry, done: int, total: int):
            """Callback invocado desde el hilo de lectura. (H1)"""
            pct = done / total * 100
            msg = f'Leyendo: {entry.name}  ({done}/{total})'
            self.after(0, lambda: [
                self._progress_var.set(pct),
                self._progress_lbl.configure(text=msg),
                self._update_entry_label(entry),
            ])

        def _worker():
            try:
                combined, updated = read_entries_parallel(entries, _on_progress)
                ok_count  = sum(1 for e in updated if e.status == 'ok')
                err_count = sum(1 for e in updated if e.status == 'error')

                if not combined.strip():
                    self.after(0, lambda: self._on_error(
                        'Ningún archivo produjo texto legible.'
                    ))
                    return

                wc = generate_wordcloud(
                    combined, max_words=max_words, bg_color=bg_color,
                    colormap=colormap, lang=lang, custom_stopwords=custom_sw,
                    width=ww, height=wh,
                )
                self.after(0, lambda: self._on_generated(wc, ok_count, err_count, combined))
            except Exception as exc:
                msg = str(exc)
                self.after(0, lambda: self._on_error(msg))

        threading.Thread(target=_worker, daemon=True).start()

    def _on_generated(self, wc: WordCloud, ok: int, err: int, combined: str):
        self._wordcloud     = wc
        self._combined_text = combined
        self._processing    = False
        self._progress_var.set(100)
        self._progress_lbl.configure(text='')

        self._placeholder.pack_forget()
        self._mpl_widget.pack(fill='both', expand=True)

        self._ax.clear()
        self._ax.imshow(wc, interpolation='bilinear')
        self._ax.axis('off')
        self._fig.tight_layout(pad=0)
        self._mpl_canvas.draw()

        n_cloud = len(wc.words_)
        total_w = len(combined.split())
        status  = f'Nube generada: {n_cloud} palabras únicas de {ok} archivo{"s" if ok!=1 else ""}'
        if err:
            status += f'  ·   {err} con error'
        self._set_status(status + '.  Guarda con Ctrl+S.', 'success' if not err else 'warning')
        self._word_count_var.set(f'{total_w:,} palabras totales · {n_cloud} en la nube')

        # H9: si hubo errores, mostrar detalle
        if err:
            errors = [f'• {e.name}: {e.error}'
                      for e in self._entries if e.status == 'error']
            messagebox.showwarning(
                f'{err} archivo{"s" if err>1 else ""} con error',
                'Los siguientes archivos no pudieron leerse:\n\n' +
                '\n'.join(errors[:10]) +
                ('\n…y más.' if len(errors) > 10 else '') +
                '\n\nLa nube se generó con los archivos restantes.',
                parent=self
            )

    def _on_error(self, msg: str):
        self._processing = False
        self._progress_lbl.configure(text='')
        messagebox.showerror('Error', msg + '\n\nVerifica los archivos seleccionados.', parent=self)
        self._set_status('Error al procesar la carpeta.', 'error')

    def save_image(self):
        if self._wordcloud is None:
            messagebox.showwarning('Sin nube', 'Genera una nube primero.', parent=self)
            return
        path = filedialog.asksaveasfilename(
            title='Guardar imagen',
            defaultextension='.png',
            filetypes=[('PNG','*.png'),('JPEG','*.jpg'),('SVG','*.svg'),('PDF','*.pdf')],
            initialfile='nube_palabras_carpeta',
        )
        if not path:
            return
        try:
            if Path(path).suffix.lower() in ('.svg', '.pdf'):
                self._fig.savefig(path, bbox_inches='tight', dpi=150)
            else:
                self._wordcloud.to_file(path)
            self._set_status(f'Imagen guardada: {Path(path).name}', 'success')
        except Exception as exc:
            messagebox.showerror('Error al guardar', str(exc), parent=self)

    def undo(self):
        """Restaura la nube anterior. (H3)"""
        if not self._history:
            self._set_status('No hay nube anterior para deshacer.', 'info')
            return
        wc = self._history.pop()
        self._wordcloud = wc
        self._ax.clear()
        self._ax.imshow(wc, interpolation='bilinear')
        self._ax.axis('off')
        self._fig.tight_layout(pad=0)
        self._mpl_canvas.draw()
        self._set_status('Nube anterior restaurada.', 'success')

    def _pick_bg(self, _=None):
        result = colorchooser.askcolor(color=self._bg_color.get(),
                                       title='Color de fondo', parent=self)
        if result[1]:
            self._bg_color.set(result[1])
            self._bg_swatch.configure(bg=result[1])

    def _set_status(self, msg: str, kind: str = 'info'):
        icons = {'info':'','success':'','error':'','warning':''}
        self._status_var.set(msg)
        self._status_icon.set(icons.get(kind, ''))


print(' FolderWordCloudApp cargada')

 FolderWordCloudApp cargada


---
##  8. Ejecutar la aplicación

In [8]:
if __name__ == '__main__':
    app = FolderWordCloudApp()
    app.mainloop()

---
##  9. Demo sin GUI — procesar carpeta de prueba

Crea archivos de muestra, los escanea y genera la nube en el notebook directamente.

In [9]:
import tempfile, os, json as _json
import matplotlib
matplotlib.use('Agg')  # sin ventana de escritorio
import matplotlib.pyplot as plt

# 1. Crear carpeta temporal con documentos de muestra
tmpdir = tempfile.mkdtemp(prefix='demo_nube_')

samples = {
    'python.txt': 'Python es un lenguaje de programación interpretado. '
                  'Python soporta programación orientada a objetos y funcional. '
                  'La comunidad Python es activa y global.',

    'data.json':  _json.dumps({
        'titulo': 'Ciencia de datos con Python',
        'temas':  ['machine learning', 'redes neuronales', 'análisis de datos',
                   'visualización', 'estadística', 'minería de datos'],
        'descripcion': 'El análisis de datos con Python incluye pandas, '
                       'numpy, scikit-learn y matplotlib.'
    }),

    'linux.txt':  'Linux es un sistema operativo de código abierto. '
                  'El kernel de Linux fue creado por Linus Torvalds. '
                  'Linux se usa en servidores, dispositivos móviles y supercomputadoras.',

    'web.csv':    'tecnología,descripción\nHTML,lenguaje de marcado para páginas web\n'
                  'CSS,estilos visuales para páginas web\n'
                  'JavaScript,lenguaje de programación para el navegador web',
}

for name, content in samples.items():
    with open(os.path.join(tmpdir, name), 'w', encoding='utf-8') as f:
        f.write(content)

print(f'Carpeta temporal: {tmpdir}')
print(f'Archivos creados: {list(samples.keys())}')

# 2. Escanear
entries = scan_folder(tmpdir)
print(f'\nArchivos detectados: {len(entries)}')
for e in entries:
    print(f'  {e.name}')

# 3. Leer en paralelo
progress_log = []
def on_prog(entry, done, total):
    progress_log.append(f'[{done}/{total}] {entry.name}  {entry.status}')

combined, updated = read_entries_parallel(entries, on_prog)
print('\nProgreso:')
for p in progress_log:
    print(' ', p)

ok  = sum(1 for e in updated if e.status == 'ok')
err = sum(1 for e in updated if e.status == 'error')
print(f'\nResultado: {ok} ok, {err} errores')
print(f'Palabras totales combinadas: {len(combined.split()):,}')

# 4. Generar nube
wc = generate_wordcloud(combined, max_words=60, lang='Español',
                         colormap='plasma', width=900, height=400)

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title(f'Demo carpeta — {ok} archivos · {len(wc.words_)} palabras únicas',
             fontsize=12, pad=10)
plt.tight_layout()
plt.show()

print(f'\nTop 10 palabras:')
for w, f in list(wc.words_.items())[:10]:
    print(f'  {w:20s}  {f:.3f}')

Carpeta temporal: /tmp/demo_nube_2u8ey61x
Archivos creados: ['python.txt', 'data.json', 'linux.txt', 'web.csv']

Archivos detectados: 4
  data.json
  linux.txt
  python.txt
  web.csv

Progreso:
  [1/4] data.json  ok
  [2/4] linux.txt  ok
  [3/4] python.txt  ok
  [4/4] web.csv  ok

Resultado: 4 ok, 0 errores
Palabras totales combinadas: 100

Top 10 palabras:
  Python                1.000
  datos                 0.800
  Linux                 0.600
  lenguaje              0.600
  programación          0.600
  web                   0.600
  análisis              0.400
  páginas               0.400
  Ciencia               0.200
  machine               0.200


/tmp/ipykernel_286323/2400362499.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

##  Resumen comparativo `4/` vs `5/`

| Característica | `4/` | `5/` |
|---------------|------|------|
| Entrada | Un archivo | Una carpeta |
| Búsqueda | — | `scan_folder()` recursiva opcional |
| Lectura | Secuencial, 1 hilo | Paralela, `ThreadPoolExecutor` |
| Progreso | Indeterminado | **Determinado** (`N/Total`) |
| Lista de archivos | — | Checkboxes con estado `  ` |
| Control (H3) | Undo | Undo + deselección por archivo |
| Errores (H9) | Por archivo | Resumen al finalizar con lista |
| Texto base | 1 documento | Concatenación de todos |
| Modelo de datos | — | `FileEntry` (dataclass) |